# Functional Programming in Python — Expert Interview Guide

Covers: closures, decorators, generators, `itertools`, `functools`, lazy evaluation, and composition.

> **Key insight:** Python is multi-paradigm. Functional tools reduce side effects and improve composability.

## 1. First-Class Functions

Functions are objects — they can be stored, passed, returned, and compared.

In [ ]:
def greet(name): return f"Hello, {name}!"
def shout(name): return f"HEY {name.upper()}!"

# Functions are objects
print(type(greet))
print(greet.__name__)

# Store in data structures
actions = {"greet": greet, "shout": shout}
print(actions["greet"]("Alice"))
print(actions["shout"]("Alice"))

# Pass as argument
def apply(fn, value): return fn(value)
print(apply(greet, "Bob"))
print(apply(len, "hello"))

# Return from function
def make_multiplier(n):
    def multiplier(x): return x * n
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5))   # 10
print(triple(5))   # 15
print(double.__name__)  # multiplier

> **Interview Insight:** Functions expose `__name__`, `__doc__`, `__annotations__`, `__defaults__`, `__code__`, `__globals__`. Use `inspect.signature()` for full introspection.

## 2. Closures

A closure is a function that **captures variables from its enclosing scope**. The captured variables are stored in `__closure__`.

In [ ]:
def make_counter(start=0):
    count = start  # free variable captured by closure

    def increment(step=1):
        nonlocal count
        count += step
        return count

    def reset():
        nonlocal count
        count = start

    def get():
        return count

    return increment, reset, get

inc, reset, get = make_counter(10)
print(inc())    # 11
print(inc(5))   # 16
print(get())    # 16
reset()
print(get())    # 10

# Inspect closure
def outer(x):
    def inner(y): return x + y
    return inner

add5 = outer(5)
print(add5.__closure__)                          # (<cell object>,)
print(add5.__closure__[0].cell_contents)         # 5

# Classic closure gotcha
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])  # [2, 2, 2] — all capture same 'i'!

# Fix: capture by default argument
funcs_fixed = [lambda i=i: i for i in range(3)]
print([f() for f in funcs_fixed])  # [0, 1, 2]

> **Interview Insight:** The `lambda: i` closure gotcha is extremely common. All lambdas capture the **same** `i` variable, not its value at creation time. Fix with `lambda i=i: i` (default arg captures current value).

## 3. Decorators

A decorator is a callable that takes a function and returns a replacement function.

In [ ]:
import functools, time

# Simple decorator
def timer(fn):
    @functools.wraps(fn)   # preserves fn.__name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIMER] {fn.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum.__name__)  # slow_sum (preserved by functools.wraps)

# Decorator with arguments — factory pattern
def retry(max_attempts=3, delay=0.1):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts: raise
                    print(f"Attempt {attempt} failed: {e}")
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=2)
def flaky(n):
    import random
    if random.random() < 0.3: raise ValueError("Random error!")
    return n * 2

print(flaky(5))

# Stacking decorators (applied bottom-up)
@timer
@retry(max_attempts=2)
def important_task():
    return "done"

result = important_task()
print(result)

# Class-based decorator (stateful)
class CallCounter:
    def __init__(self, fn):
        functools.update_wrapper(self, fn)
        self.fn = fn
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.fn(*args, **kwargs)

@CallCounter
def greet(name): return f"Hi {name}"

greet("Alice")
greet("Bob")
print(f"greet called {greet.count} times")

> **Interview Insight:** Always use `@functools.wraps(fn)` in decorators — without it, `fn.__name__`, `fn.__doc__`, and `help()` show the wrapper, not the original function. This breaks debugging and introspection.

## 4. Generators: `yield`, `yield from`, `send()`, `throw()`

Generators are lazy iterators that produce values one at a time.

In [ ]:
import sys

# Basic generator
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print([next(fib) for _ in range(10)])

# Generator is a lazy iterator
def squares(n):
    for i in range(n):
        yield i ** 2

gen = squares(5)
print(type(gen))
print(sys.getsizeof(gen))    # tiny — just the generator frame
for val in gen: print(val, end=" ")
print()

# yield from — delegate to sub-generator
def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1, 2], [3, 4], [5, 6])))

# Generator pipeline
def read_data():
    for x in range(10): yield x

def filter_even(data):
    for x in data:
        if x % 2 == 0: yield x

def square_it(data):
    for x in data: yield x ** 2

pipeline = square_it(filter_even(read_data()))
print(list(pipeline))  # [0, 4, 16, 36, 64]

# send() — two-way communication
def accumulator():
    total = 0
    while True:
        value = yield total  # yield current total, receive new value
        if value is None: break
        total += value

acc = accumulator()
next(acc)           # prime the generator (advance to first yield)
acc.send(10)
acc.send(20)
result = acc.send(30)
print(f"Accumulator total: {result}")  # 60

# Generator as context manager
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    try:
        yield name.upper()
    finally:
        print(f"Releasing {name}")

with managed_resource("database") as res:
    print(f"Using: {res}")

> **Interview Insight:** You must `next(gen)` (or `gen.send(None)`) to advance to the first `yield` before calling `gen.send(value)`. Calling `send(value)` on an un-started generator raises `TypeError`.

## 5. `itertools` — Essential Tools

In [ ]:
import itertools

# chain — flatten iterables
print(list(itertools.chain([1,2], [3,4], [5])))     # [1,2,3,4,5]
print(list(itertools.chain.from_iterable([[1,2],[3,4]])))  # same

# islice — lazy slice of iterator
gen = (x**2 for x in range(100))
print(list(itertools.islice(gen, 5)))  # [0,1,4,9,16]

# count, cycle, repeat
counter = itertools.count(start=10, step=5)
print([next(counter) for _ in range(4)])  # [10,15,20,25]

cycler = itertools.cycle(["a","b","c"])
print([next(cycler) for _ in range(7)])   # ['a','b','c','a','b','c','a']

print(list(itertools.repeat(42, 3)))      # [42,42,42]

# product — cartesian product
print(list(itertools.product([1,2], ["a","b"])))

# permutations / combinations
print(list(itertools.permutations("ABC", 2)))  # ordered pairs
print(list(itertools.combinations("ABC", 2)))  # unordered pairs

# groupby — consecutive groups (data must be sorted first!)
data = [("apple","fruit"),("banana","fruit"),("carrot","veg"),("daikon","veg")]
for key, group in itertools.groupby(data, key=lambda x: x[1]):
    print(f"{key}: {[x[0] for x in group]}")

# accumulate — running totals
import operator
nums = [1, 2, 3, 4, 5]
print(list(itertools.accumulate(nums)))            # [1,3,6,10,15]
print(list(itertools.accumulate(nums, operator.mul)))  # [1,2,6,24,120]

# takewhile / dropwhile
print(list(itertools.takewhile(lambda x: x < 5, [1,2,3,4,5,6])))  # [1,2,3,4]
print(list(itertools.dropwhile(lambda x: x < 5, [1,2,3,4,5,6])))  # [5,6]

# starmap
print(list(itertools.starmap(pow, [(2,3),(3,2),(4,1)])))  # [8,9,4]

> **Interview Insight:** `groupby` groups **consecutive** equal elements — like Unix `uniq`. Always sort by the key first. Unlike SQL GROUP BY, it won't collect all matching items if they're not adjacent.

## 6. `functools` — Higher-Order Function Tools

In [ ]:
import functools, operator

# reduce — fold left
print(functools.reduce(operator.add, [1,2,3,4,5]))  # 15
print(functools.reduce(lambda acc, x: acc * x, [1,2,3,4,5]))  # 120

# partial — freeze arguments
def power(base, exp): return base ** exp

square = functools.partial(power, exp=2)
cube   = functools.partial(power, exp=3)
print(square(4), cube(3))  # 16, 27

# lru_cache — memoization
@functools.lru_cache(maxsize=None)
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(fib(50))
print(fib.cache_info())   # hits, misses, maxsize, currsize

# cache (Python 3.9+) — unlimited LRU
@functools.cache
def expensive(n):
    return sum(range(n))

print(expensive(1000))
print(expensive(1000))  # from cache

# cached_property — computed once, cached on instance
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @functools.cached_property
    def area(self):
        import math
        print("  [Computing area...]")
        return math.pi * self.radius ** 2

c = Circle(5)
print(c.area)  # computes
print(c.area)  # cached — no recomputation

# total_ordering — fill in comparison methods
@functools.total_ordering
class Card:
    RANKS = "23456789TJQKA"
    def __init__(self, rank): self.rank = rank
    def __eq__(self, other): return self.rank == other.rank
    def __lt__(self, other):
        return self.RANKS.index(self.rank) < self.RANKS.index(other.rank)

cards = [Card("A"), Card("2"), Card("K"), Card("7")]
print([c.rank for c in sorted(cards)])  # ['2','7','K','A']

# singledispatch — function overloading by type
@functools.singledispatch
def process(value):
    raise TypeError(f"Unsupported type: {type(value)}")

@process.register(int)
def _(value): return f"int: {value * 2}"

@process.register(str)
def _(value): return f"str: {value.upper()}"

@process.register(list)
def _(value): return f"list: {len(value)} items"

print(process(5))
print(process("hello"))
print(process([1,2,3]))

> **Interview Insight:** `@functools.lru_cache` only works on functions with **hashable** arguments. If you need to cache with unhashable args (list, dict), convert to tuple/frozenset first.

## 7. Lazy Evaluation & Memory Efficiency

In [ ]:
import sys

# Generator expression vs list comprehension
nums = range(1_000_000)

list_comp = [x**2 for x in nums]        # eager — stores all 1M values
gen_expr  = (x**2 for x in nums)        # lazy — stores nothing

print(f"List size: {sys.getsizeof(list_comp):,} bytes")
print(f"Gen size:  {sys.getsizeof(gen_expr)} bytes")

# Process large file lazily
def read_lines_lazy(filename, encoding="utf-8"):
    # Generator — only one line in memory at a time
    with open(filename, encoding=encoding) as f:
        for line in f:
            yield line.rstrip()

# Pipeline: filter + transform + limit — all lazy
def process_pipeline(source):
    filtered = (line for line in source if line)
    uppercased = (line.upper() for line in filtered)
    return uppercased

# Demonstrate with in-memory "file"
lines = ["hello", "", "world", "", "python"]
result = list(process_pipeline(iter(lines)))
print(result)

# Infinite lazy sequences
def naturals(start=1):
    n = start
    while True:
        yield n
        n += 1

def take(n, iterable):
    import itertools
    return list(itertools.islice(iterable, n))

print(take(10, naturals()))
print(take(10, (x**2 for x in naturals())))

# Lazy vs eager — when each is better
# Lazy (generator): streaming data, unknown size, only need one pass
# Eager (list): random access, multiple passes, need length, small data

> **Interview Insight:** A generator expression `(...)` is just syntactic sugar for a generator function with `yield`. Once exhausted, generators cannot be restarted — you must create a new one.

## 8. Function Composition & Currying

In [ ]:
import functools
from typing import Callable

# Function composition: h(x) = f(g(x))
def compose(*fns):
    def composed(x):
        for fn in reversed(fns):
            x = fn(x)
        return x
    return composed

double   = lambda x: x * 2
add_one  = lambda x: x + 1
square   = lambda x: x ** 2

# right-to-left: square first, then add_one, then double
transform = compose(double, add_one, square)
print(transform(3))  # double(add_one(square(3))) = double(add_one(9)) = double(10) = 20

# Pipe (left-to-right composition)
def pipe(*fns):
    return functools.reduce(lambda f, g: lambda x: g(f(x)), fns)

process = pipe(square, add_one, double)
print(process(3))  # double(add_one(square(3))) = 20

# Currying — convert f(a,b,c) to f(a)(b)(c)
def curry(fn):
    import inspect
    n_args = len(inspect.signature(fn).parameters)

    def curried(*args):
        if len(args) >= n_args:
            return fn(*args)
        return lambda *more: curried(*(args + more))
    return curried

@curry
def add(a, b, c): return a + b + c

print(add(1)(2)(3))        # 6
print(add(1, 2)(3))        # 6
print(add(1)(2, 3))        # 6
add1 = add(1)              # partial application
add1_2 = add(1, 2)
print(add1_2(10))          # 13

> **Interview Insight:** `functools.partial` is Python's built-in partial application (not full currying). For true currying, you need a custom implementation. Composition + currying are the core tools of functional pipelines.